In [3]:
# This notebook trains models using a single 80/20 split with wealth-exclusive features
import numpy as np
import pandas as pd

df = pd.read_parquet("PAKISTAN/DATA/6.PK_TRAIN_80SET.parquet", engine="pyarrow")

In [ ]:
#normalize to numeric first
cols_to_numeric = [
    "hv024RegionDivision",
    "hv025TypeOfResidence",
    "hv219SexOfHead",
    "hv270WealthIndex",  
    "hv106_01EducationOfHead",
    "hv115_01MaritalStatus",
    "v717Occupation",
    "745aHouseOwnership",
]
for col in cols_to_numeric:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [4]:
def restore_ml_dtypes(df):
    df = df.copy()

    df["hv220AgeOfHead"] = df["hv220AgeOfHead"].astype("int8")
    df["hv009FamilySize"] = df["hv009FamilySize"].astype("int8")
    df["hv216HouseSize"] = df["hv216HouseSize"].astype("float64")
    df["hv014NoOfChildren"] = df["hv014NoOfChildren"].astype("int8")

    df[[
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv270WealthIndex",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "v717Occupation",
        "745aHouseOwnership",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
    ]] = df[[
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv270WealthIndex",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "v717Occupation",
        "745aHouseOwnership",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
    ]].astype("category")

    df[[
        "EnergyPoor",
    ]] = df[[
        "EnergyPoor",
    ]].astype("int8")

    return df

In [5]:
df_restore = restore_ml_dtypes(df)

In [6]:
print("\nDtypes:")
print(df_restore.dtypes)

print("\nMissing values:")
print(df_restore.isna().sum().sort_values(ascending=False).head())


Dtypes:
hv270WealthIndex           category
hv024RegionDivision        category
hv025TypeOfResidence       category
hv219SexOfHead             category
hv220AgeOfHead                 int8
hv106_01EducationOfHead    category
hv115_01MaritalStatus      category
hv009FamilySize                int8
hv247HasBankAccount        category
hv216HouseSize                 int8
hv014NoOfChildren              int8
v714CurrentlyWorking       category
v717Occupation             category
745aHouseOwnership         category
EnergyPoor                     int8
dtype: object

Missing values:
hv270WealthIndex        0
hv024RegionDivision     0
hv025TypeOfResidence    0
hv219SexOfHead          0
hv220AgeOfHead          0
dtype: int64


In [7]:
TARGET = "EnergyPoor"

drop_cols_mepi_wealth_exclusive = [

    # Per-country model
    "hv000CountryCode",

    # ID variables
    "hhidCaseIdentification",
    "hv001ClusterNumber",
    "hv002HouseholdNumber",

    # Sampling weights
    "hv005SimplilingWeight",
    "weight",

    # Wealth index (core difference)
    "hv270WealthIndex",
] 

# Build clean training dataframe
X_cols = [c for c in df_restore.columns
          if c not in drop_cols_mepi_wealth_exclusive + [TARGET]]

train_df = df_restore[X_cols + [TARGET]]

print("WEALTH-EXCLUSIVE SPECIFICATION")
print("Target:", TARGET)
print("Number of features:", len(X_cols))
print("Features used:", X_cols)


WEALTH-EXCLUSIVE SPECIFICATION
Target: EnergyPoor
Number of features: 13
Features used: ['hv024RegionDivision', 'hv025TypeOfResidence', 'hv219SexOfHead', 'hv220AgeOfHead', 'hv106_01EducationOfHead', 'hv115_01MaritalStatus', 'hv009FamilySize', 'hv247HasBankAccount', 'hv216HouseSize', 'hv014NoOfChildren', 'v714CurrentlyWorking', 'v717Occupation', '745aHouseOwnership']


In [8]:
print("\nDtypes:")
print(train_df.dtypes)

print("\nMissing values:")
print(train_df.isna().sum().sort_values(ascending=False).head())


Dtypes:
hv024RegionDivision        category
hv025TypeOfResidence       category
hv219SexOfHead             category
hv220AgeOfHead                 int8
hv106_01EducationOfHead    category
hv115_01MaritalStatus      category
hv009FamilySize                int8
hv247HasBankAccount        category
hv216HouseSize                 int8
hv014NoOfChildren              int8
v714CurrentlyWorking       category
v717Occupation             category
745aHouseOwnership         category
EnergyPoor                     int8
dtype: object

Missing values:
hv024RegionDivision        0
hv025TypeOfResidence       0
hv219SexOfHead             0
hv220AgeOfHead             0
hv106_01EducationOfHead    0
dtype: int64


In [9]:
from autogluon.tabular import TabularPredictor

LABEL = "EnergyPoor"

MODEL_PATH = "PAKISTAN/WEALTH_EXCLUSIVE/TRADITIONAL/"

hyperparameters_tr = {
    "GBM": {},   # one LightGBM only
    "CAT": {},   # one CatBoost
    "RF": {},    # one Random Forest
    "XT": {}     # one Extra Trees
}

predictor_tr = TabularPredictor(
    label=LABEL,
    problem_type="binary",
    eval_metric="accuracy",
    path=MODEL_PATH,
    verbosity=2
).fit(
    train_data=train_df,
    presets="medium_quality",
    hyperparameters=hyperparameters_tr,
    fit_weighted_ensemble=False,
    num_stack_levels=0
)

C:\Users\ayeei\.conda\envs\ag_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.19
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          22
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       10.42 GB / 31.46 GB (33.1%)
Disk Space Avail:   245.27 GB / 350.00 GB (70.1%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ...
AutoGluon will save models to "D:\PERSONAL\Otago\INFO501\MyChoice\ReferencePaper\Data\DHSData\PROCESSING\REPO\PAKISTAN\WEALTH_EXCLUSIVE\TRADITIONAL"
Train Data Rows:    11556
Train Data Columns: 13
Label Column:       EnergyPoor
Problem Type:       binary


In [10]:
from autogluon.tabular import TabularPredictor
from tabnet_ag_model import NeuralNetTabNet

LABEL = "EnergyPoor"

DL_MODEL_PATH = "PAKISTAN/WEALTH_EXCLUSIVE/DL/"

hyperparameters_dl = {
    "NN_TORCH": {},   # PyTorch tabular NN
    "FASTAI": {},     # fastai tabular NN
    NeuralNetTabNet: [{}]  # custom TabNet
}

predictor_dl = TabularPredictor(
    label=LABEL,
    problem_type="binary",
    eval_metric="accuracy",
    path=DL_MODEL_PATH,
    verbosity=2
).fit(
    train_data=train_df,
    presets="medium_quality",
    hyperparameters=hyperparameters_dl,
    fit_weighted_ensemble=False,
    num_stack_levels=0
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.19
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          22
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       10.12 GB / 31.46 GB (32.2%)
Disk Space Avail:   245.03 GB / 350.00 GB (70.0%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ...
AutoGluon will save models to "D:\PERSONAL\Otago\INFO501\MyChoice\ReferencePaper\Data\DHSData\PROCESSING\REPO\PAKISTAN\WEALTH_EXCLUSIVE\DL"
Train Data Rows:    11556
Train Data Columns: 13
Label Column:       EnergyPoor
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    10368.89 MB
	Train Data (Original)  Memory Usage: 


Early stopping occurred at epoch 48 with best_epoch = 28 and best_val_0_auc = 0.85932


C:\Users\ayeei\.conda\envs\ag_env\lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
	0.7829	 = Validation score   (accuracy)
	47.19s	 = Training   runtime
	0.05s	 = Validation runtime
AutoGluon training complete, total runtime = 81.29s ... Best model: NeuralNetTorch | Estimated inference throughput: 63170.0 rows/s (1156 batch size)
Disabling decision threshold calibration for metric `accuracy` due to having fewer than 10000 rows of validation data for calibration, to avoid overfitting (1156 rows).
	`accuracy` is generally not improved through threshold calibration. Force calibration via specifying `calibrate_decision_threshold=True`.
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("D:\PERSONAL\Otago\INFO501\MyChoice\ReferencePaper\Data\DHSData\PROCESSING\REPO\PAKISTAN\WEALTH_EXCLUSIVE\DL")


In [11]:
from autogluon.tabular import TabularPredictor
from FTTransformer_ag_model import NeuralNetFTTransformer

LABEL = "EnergyPoor"

DLFT_MODEL_PATH = "PAKISTAN/WEALTH_EXCLUSIVE/DLFT"

hyperparameters_ft = {
      NeuralNetFTTransformer: [{
         "use_cuda_if_available": True
     }]  # custom FT-Transformer
}

predictor_ft = TabularPredictor(
    label=LABEL,
    problem_type="binary",
    eval_metric="accuracy",
    path=DLFT_MODEL_PATH,
    verbosity=2
).fit(
    train_data=train_df,
    presets="medium_quality",
    hyperparameters=hyperparameters_ft,
    fit_weighted_ensemble=False,
    num_stack_levels=0,
    num_gpus=1,      # <-- required
    num_cpus=8
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.19
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          22
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       10.10 GB / 31.46 GB (32.1%)
Disk Space Avail:   245.03 GB / 350.00 GB (70.0%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ...
AutoGluon will save models to "D:\PERSONAL\Otago\INFO501\MyChoice\ReferencePaper\Data\DHSData\PROCESSING\REPO\PAKISTAN\WEALTH_EXCLUSIVE\DLFT"
Train Data Rows:    11556
Train Data Columns: 13
Label Column:       EnergyPoor
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    10328.48 MB
	Train Data (Original)  Memory Usage